# PathSnap: 5-Model Experimentation & Comparison
**UE24CS352A - Machine Learning Hackathon | Mandatory 5-Model Comparison**

This notebook compares five distinct algorithm families on candidate features generated by the Member 1 pipeline:
1. **Linear and Probabilistic:** Logistic Regression (with `StandardScaler`)
2. **Tree-Based:** Decision Tree Classifier
3. **Ensemble (Bagging):** Random Forest Classifier
4. **Instance-Based:** K-Nearest Neighbors
5. **Boosting:** XGBoost Classifier

---
### Evaluation Protocols
- Five-fold `GroupKFold` keeps all observations from a trip in one fold.
- CV is the primary evaluation. Student-test metrics are computed only when that dataset explicitly contains `is_match` labels.
- Unlabeled student-test data is used for inference timing only; evaluator data must not be used for model selection.
- Scores describe only the trips present in the processed files. Use the complete training partition for final comparisons.


In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss, confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("All modeling libraries loaded successfully.")


## 1. Load Processed Candidate Dataset
Loading candidate features extracted by the Member 1 pipeline.


In [ ]:
project_root = Path.cwd()
while not (project_root / "pyproject.toml").is_file() and project_root != project_root.parent:
    project_root = project_root.parent

processed_dir = project_root / "src" / "pathsnap" / "data" / "processed"
train_path = processed_dir / "train_candidates.parquet"
test_path = processed_dir / "student_test_candidates.parquet"
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(
        f"Processed candidates not found in {processed_dir}. "
        "Run `python scripts/build_dataset.py` from the repository root first."
    )

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
test_has_labels = "is_match" in test_df.columns

FEATURE_COLS = [
    "dist_perp",
    "dist_to_start",
    "dist_to_end",
    "arc_length",
    "heading_diff",
    "speed",
    "proj_ratio"
]

print(f"Training candidates: {len(train_df):,} rows across {train_df['trip_id'].nunique()} trips")
print(f"Student-test candidates: {len(test_df):,} rows across {test_df['trip_id'].nunique()} trips")
print("Student-test labels available:", test_has_labels)
print("Target distribution (is_match in train):")
print(train_df["is_match"].value_counts(normalize=True))


## 2. Evaluation Utilities
Defining GroupKFold cross-validation, observation-level Top-1 accuracy, and metric computation.


In [ ]:
def compute_observation_top1(df_subset, probas):
    df_eval = df_subset[["trip_id", "obs_id", "is_match"]].copy()
    df_eval["score"] = probas
    idx_max = df_eval.groupby(["trip_id", "obs_id"])["score"].idxmax()
    return float(df_eval.loc[idx_max, "is_match"].mean())

def run_group_cv(model, train_df, n_splits=5):
    group_count = train_df["trip_id"].nunique()
    if group_count < n_splits:
        raise ValueError(f"GroupKFold needs {n_splits} distinct trips; found {group_count}.")

    gkf = GroupKFold(n_splits=n_splits)
    X = train_df[FEATURE_COLS].values
    y = train_df["is_match"].values
    groups = train_df["trip_id"].values

    cv_f1, cv_top1, cv_loss = [], [], []
    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        val_df = train_df.iloc[val_idx]

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]
        preds = (probs >= 0.5).astype(int)
        cv_f1.append(f1_score(y_val, preds, average="macro", zero_division=0))
        cv_top1.append(compute_observation_top1(val_df, probs))
        cv_loss.append(log_loss(y_val, np.clip(probs, 1e-7, 1 - 1e-7)))

    return np.mean(cv_f1), np.std(cv_f1), np.mean(cv_top1)

print("Evaluation pipeline initialized.")


## 3. Train & Evaluate the 5 Models Across Algorithm Families
We evaluate:
1. **Logistic Regression** (Linear & Probabilistic)
2. **Decision Tree** (Tree-based)
3. **Random Forest** (Ensemble)
4. **KNN** (Instance-based)
5. **XGBoost** (Boosting)


In [ ]:
models = {
    "Logistic Regression": (
        "Linear & Probabilistic",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42))
        ]),
        "C=1.0, StandardScaler, balanced"
    ),
    "Decision Tree": (
        "Tree-based",
        DecisionTreeClassifier(max_depth=10, min_samples_split=20, min_samples_leaf=10, class_weight="balanced", random_state=42),
        "max_depth=10, min_samples_leaf=10, balanced"
    ),
    "Random Forest": (
        "Ensemble",
        RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_split=15, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=42),
        "n_estimators=100, max_depth=12, balanced, n_jobs=-1"
    ),
    "KNN": (
        "Instance-based",
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=7, weights="distance", algorithm="kd_tree", n_jobs=-1))
        ]),
        "k=7, distance-weighted, StandardScaler"
    ),
    "XGBoost": (
        "Boosting",
        xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=9.0, eval_metric="logloss", random_state=42, n_jobs=-1),
        "n_est=100, depth=6, lr=0.1, scale_pos_weight=9.0"
    )
}

results = []
test_predictions = {}
X_train, y_train = train_df[FEATURE_COLS].values, train_df["is_match"].values
X_test = test_df[FEATURE_COLS].values
y_test = test_df["is_match"].values if test_has_labels else None

for name, (family, model, config) in models.items():
    print(f"Training & evaluating {name} ({family})...")
    cv_f1_mean, cv_f1_std, cv_top1_mean = run_group_cv(model, train_df)

    model.fit(X_train, y_train)
    t0 = time.time()
    probs = model.predict_proba(X_test)[:, 1]
    inf_time_ms = (time.time() - t0) / len(X_test) * 1e6
    preds = (probs >= 0.5).astype(int)
    test_predictions[name] = {"probs": probs, "preds": preds}

    if test_has_labels:
        test_f1 = f1_score(y_test, preds, average="macro", zero_division=0)
        test_top1 = compute_observation_top1(test_df, probs)
        test_loss = log_loss(y_test, np.clip(probs, 1e-7, 1 - 1e-7))
        test_f1_text, test_top1_text, test_loss_text = f"{test_f1:.4f}", f"{test_top1:.4f}", f"{test_loss:.4f}"
    else:
        test_f1_text = test_top1_text = test_loss_text = "N/A (unlabeled)"

    results.append({
        "Model": name,
        "Algorithm Family": family,
        "Key Configuration": config,
        "CV Macro F1": f"{cv_f1_mean:.4f} (+/- {cv_f1_std:.4f})",
        "CV Top-1 Accuracy": f"{cv_top1_mean:.4f}",
        "Top-1 Accuracy": test_top1_text,
        "Route continuity": "N/A (Classifier only)",
        "Student-Test Macro F1-score": test_f1_text,
        "Log Loss": test_loss_text,
        "Inference Time": f"{inf_time_ms:.2f} ms/1k"
    })

df_results = pd.DataFrame(results)
results_dir = project_root / "reports" / "generated"
results_dir.mkdir(parents=True, exist_ok=True)
df_results.to_csv(results_dir / "all_5_models_results.csv", index=False)
display(df_results)


## 4. Visualizations (Mandatory Section 3.3)
### 4.1 Confusion Matrices for All 5 Models


In [ ]:
if test_has_labels:
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    for idx, (name, res) in enumerate(test_predictions.items()):
        cm = confusion_matrix(y_test, res["preds"], normalize="true")
        sns.heatmap(cm, annot=True, fmt=".1%", cmap="Blues", ax=axes[idx], cbar=False,
                    xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
        axes[idx].set_title(name, fontweight="bold", fontsize=12)
        axes[idx].set_xlabel("Predicted")
        axes[idx].set_ylabel("True" if idx == 0 else "")
    plt.suptitle("Student-Test Confusion Matrices", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrices skipped: student-test data has no labels.")


### 4.2 Overall Performance Comparison Across 5 Models
Comparing Observation-Level Top-1 Accuracy and Macro F1 on the Student Testing set.


In [ ]:
metric_label = "Student-Test" if test_has_labels else "GroupKFold CV"
top1_key = "Top-1 Accuracy" if test_has_labels else "CV Top-1 Accuracy"
f1_key = "Student-Test Macro F1-score" if test_has_labels else "CV Macro F1"
top1_scores = [float(row[top1_key]) for row in results]
f1_scores = [float(row[f1_key].split()[0]) for row in results]
names = [row["Model"] for row in results]

plt.figure(figsize=(12, 5))
x = np.arange(len(names))
width = 0.35
plt.bar(x - width/2, top1_scores, width, label=f"{metric_label} Top-1 Accuracy", color="#2ecc71")
plt.bar(x + width/2, f1_scores, width, label=f"{metric_label} Macro F1", color="#3498db")
plt.xticks(x, names, fontweight="bold", fontsize=11)
plt.ylabel("Score", fontsize=12)
plt.ylim(0, 1.0)
plt.title(f"PathSnap 5-Model Comparison ({metric_label})", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=11)
for i in range(len(names)):
    plt.text(i - width/2, top1_scores[i] + 0.008, f"{top1_scores[i]:.1%}", ha="center", fontweight="bold", fontsize=10)
    plt.text(i + width/2, f1_scores[i] + 0.008, f"{f1_scores[i]:.1%}", ha="center", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.savefig(results_dir / "all_5_models_comparison.png", dpi=300)
plt.show()


### 4.3 Feature Importance Comparison (Random Forest vs XGBoost)
Comparing Gini Importance from Bagging (Random Forest) vs Split Gain from Boosting (XGBoost).


In [ ]:
rf_model = models["Random Forest"][1]
xgb_model = models["XGBoost"][1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Random Forest
rf_imp = rf_model.feature_importances_
idx_rf = np.argsort(rf_imp)[::-1]
ax1.bar(range(len(FEATURE_COLS)), rf_imp[idx_rf], color="#e67e22")
ax1.set_xticks(range(len(FEATURE_COLS)))
ax1.set_xticklabels([FEATURE_COLS[i] for i in idx_rf], rotation=30, ha="right", fontweight="bold")
ax1.set_title("Random Forest Feature Importances", fontweight="bold", fontsize=13)
ax1.set_ylabel("Gini Importance")

# XGBoost
xgb_imp = xgb_model.feature_importances_
idx_xgb = np.argsort(xgb_imp)[::-1]
ax2.bar(range(len(FEATURE_COLS)), xgb_imp[idx_xgb], color="#9b59b6")
ax2.set_xticks(range(len(FEATURE_COLS)))
ax2.set_xticklabels([FEATURE_COLS[i] for i in idx_xgb], rotation=30, ha="right", fontweight="bold")
ax2.set_title("XGBoost Feature Importances", fontweight="bold", fontsize=13)
ax2.set_ylabel("Weight / Gain")

plt.tight_layout()
plt.show()


## 5. Model Selection Technical Justification
Use the trip-grouped cross-validation results as the primary model-selection evidence.
Student-test scores are shown only when the processed student-test data includes labels;
do not use evaluator data for model selection. Re-run after building the full training partition.
